# Class 7: Building and evaluating a classifier for movement expressions

This notebook mirrors sections 3–7 of the [Class 7 page](https://nlp.lang.osaka-u.ac.jp/dh/b/2026-11-24/). Run each cell from the top and edit the six cells marked **Edit**.

**AI policy by part:** In sections 3–6, you may ask an assistant to explain Python error messages. Do not use an assistant to predict outcomes before running a cell, or to read and judge passages in sections 3, 4, and 7.

### Setup (run-only)
Run this collapsed cell once. It downloads the Class 7 bundle and loads plain Python lists (`train_rows`, `dev_rows`, `test_rows`) and helper functions. You do not need to read or edit its code.

In [ ]:
#@title Load passages, labels, vectors, and helpers (run-only)
import json
from collections import Counter
from urllib.request import urlretrieve
from zipfile import ZipFile

import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

LABELS = ["CT+", "CT-", "Uu", "NM"]
url = "https://nlp.lang.osaka-u.ac.jp/dh/b/materials/movement-classification.zip"
archive, _ = urlretrieve(url)

with ZipFile(archive) as bundle:
    all_passages = [json.loads(line) for line in bundle.open("class7/passages.jsonl")]
    raw_labels = [json.loads(line) for line in bundle.open("class7/llm-labels.jsonl")]
    llm_labels = {row["id"]: row["label"] for row in raw_labels if row["label"] in LABELS}
    with np.load(bundle.open("class7/embeddings.npz")) as saved:
        vec_map = dict(zip(saved["ids"].tolist(), saved["vectors"].astype("float32")))
    saved_predictions = {}
    for method in ("majority", "rules", "gliner", "gliner-ft", "zeroshot-2b", "lora-2b"):
        lines = [json.loads(line) for line in bundle.open(f"class7/predictions/{method}.jsonl")]
        saved_predictions[method] = {row["id"]: row["pred"] for row in lines[1:]}

for row in all_passages:
    row["reference"] = llm_labels.get(row["id"])

train_rows = [row for row in all_passages if row["split"] == "train" and row["reference"]]
dev_rows = [row for row in all_passages if row["split"] == "dev" and row["reference"]]
test_rows = [row for row in all_passages if row["split"] == "test" and row["reference"]]
by_id = {row["id"]: row for row in all_passages}

train_vectors = np.stack([vec_map[row["id"]] for row in train_rows])
dev_vectors = np.stack([vec_map[row["id"]] for row in dev_rows])
test_vectors = np.stack([vec_map[row["id"]] for row in test_rows])
train_labels = [row["reference"] for row in train_rows]
dev_labels = [row["reference"] for row in dev_rows]
test_labels = [row["reference"] for row in test_rows]

lemma_votes = {}
for row in train_rows:
    lemma_votes.setdefault(row["lemma"], Counter())[row["reference"]] += 1
lemma_majority = {lemma: counts.most_common(1)[0][0] for lemma, counts in lemma_votes.items()}

_order = np.random.default_rng(20261124).permutation(len(train_rows))
shuffled_vectors = train_vectors[_order]
shuffled_labels = [train_labels[i] for i in _order]

AUDIT_IDS = ("train-0407", "train-0569", "train-0360")
AUDIT_NOTES = {
    "train-0407": ("Uu", "The man looked around to see whether money had fallen. The outcome is open."),
    "train-0569": ("CT-", "来ない describes this occasion. The next sentence recalls earlier visits upstairs."),
    "train-0360": ("defer", "The reader deferred on the extracted 降. Marking the whole verb 降懸る gives CT+ for the insects descending."),
}

def show_passage(row):
    print(f'{row["author"]} 「{row["title"]}」 sentence {row["sentence"]} ({row["id"]})')
    print(row["previous"])
    print(row["left"] + "【" + row["target"] + "】" + row["right"])
    print(row["next"])

def show_audit_comparison(passage_id):
    reader_label, note = AUDIT_NOTES[passage_id]
    print(f"{passage_id} — Qwen: {by_id[passage_id]['reference']} | Reader (with AI discussion): {reader_label}")
    print(" ", note)

def saved_list(method, rows):
    return [saved_predictions[method][row["id"]] for row in rows]

def count_agreements(gold, predicted):
    matches = 0
    for g, p in zip(gold, predicted):
        if g == p:
            matches += 1
    return matches

def show_scores(name, gold, predicted):
    matches = count_agreements(gold, predicted)
    f1s = []
    recalls = {}
    for label in LABELS:
        tp = sum(g == label and p == label for g, p in zip(gold, predicted))
        n_pred = sum(p == label for p in predicted)
        n_gold = sum(g == label for g in gold)
        prec = tp / n_pred if n_pred else 0.0
        rec = tp / n_gold if n_gold else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        recalls[label] = (tp, n_gold)
        if n_gold:
            f1s.append(f1)
    macro_f1 = sum(f1s) / len(f1s) if f1s else 0.0
    nm_tp, nm_n = recalls["NM"]
    uu_tp, uu_n = recalls["Uu"]
    print(f"{name:<32} {matches:>3}/{len(gold)} | macro-F1 {macro_f1:.3f} | NM {nm_tp}/{nm_n} | Uu {uu_tp}/{uu_n}")

def show_confusion_matrix(gold, predicted):
    counts = Counter(zip(gold, predicted))
    header = "Ref \\ Pred | " + " | ".join(f"{label:>4}" for label in LABELS)
    print(header)
    print("-" * len(header))
    for g in LABELS:
        cells = " | ".join(f"{counts[(g, p)]:>4}" for p in LABELS)
        print(f"{g:>10} | {cells}")

def show_top_cues(vectorizer, model, label, k=8):
    classes = list(model.classes_)
    feats = vectorizer.get_feature_names_out()
    weights = model.coef_[classes.index(label)]
    order = np.argsort(weights)[::-1][:k]
    items = [f"{feats[i]} (+{weights[i]:.2f})" for i in order if weights[i] > 0]
    print(f"Top {label} cues:", ", ".join(items))

def test_sentence(model, vectorizer, sentence, window=10):
    start = sentence.find("【")
    end = sentence.find("】", start + 1) if start >= 0 else -1
    if start >= 0 and end > start + 1:
        clipped = sentence[:start][-window:] + sentence[start:end + 1] + sentence[end + 1:][:window] if window > 0 else sentence[start:end + 1]
    else:
        clipped = sentence
    x = vectorizer.transform([clipped])
    classes = list(model.classes_)
    probs = model.predict_proba(x)[0]
    best = int(np.argmax(probs))
    pred = classes[best]
    prob_text = ", ".join(f"{label}: {probs[classes.index(label)] * 100:.0f}%" for label in LABELS if label in classes)
    feats = vectorizer.get_feature_names_out()
    contribs = [(float(x[0, j] * model.coef_[best, j]), feats[j]) for j in x.nonzero()[1]]
    top_pos = [f"{gram} (+{w:.2f})" for w, gram in sorted(contribs, reverse=True)[:4] if w > 0.005]
    print("Window:", clipped)
    print("Prediction:", pred, "|", prob_text)
    print("Top cues:", ", ".join(top_pos))

def build_hybrid_features(rows, vectors, encoder=None):
    cue_dicts = []
    for row in rows:
        right = row["right"]
        cue_dicts.append({
            "lemma": row["lemma"],
            "negative": right.startswith(("ない", "なかっ")),
            "conditional": right.startswith(("なら", "ば", "たら")),
        })
    if encoder is None:
        encoder = DictVectorizer(sparse=False)
        cue_matrix = encoder.fit_transform(cue_dicts)
    else:
        cue_matrix = encoder.transform(cue_dicts)
    return np.column_stack((vectors, cue_matrix)), encoder

print(f"Loaded {len(train_rows)} train, {len(dev_rows)} validation, {len(test_rows)} test passages.")

### Section 3. Verify the work split and check three training labels
Each work must appear in only one split so the classifier cannot reuse work-specific names. Run the first cell to confirm zero overlap between training and test.

In [ ]:
train_works = set()
for row in train_rows:
    train_works.add(row["file"])

test_works = set()
for row in test_rows:
    test_works.add(row["file"])

shared_works = train_works & test_works
print("Training works:", len(train_works))
print("Test works:", len(test_works))
print("Works in both splits:", len(shared_works))

Read the three training passages below. Record your own label (`CT+`, `CT-`, `Uu`, `NM`, or defer on the marked span) and quote the supporting words before running the comparison cell.

In [ ]:
for passage_id in AUDIT_IDS:
    show_passage(by_id[passage_id])
    print()

In [ ]:
for passage_id in AUDIT_IDS:
    show_audit_comparison(passage_id)

### Section 4. Predict from a lemma and test a left-context rule
Read `dev-0054` below. What falls? Record your own label (`CT+`, `CT-`, `Uu`, or `NM`) before running the rule.

In [ ]:
show_passage(by_id["dev-0054"])

**Edit 1.** Start from the per-lemma majority, then switch `落ちる` to `NM` when `left` ends with `cue`. Run the next two cells with `cue = "眠りに"`, then replace `cue` with a phrase from `dev-0054` ending in `が` and rerun both cells to see how validation agreements change across all 59 passages.

In [ ]:
cue = "眠りに"

In [ ]:
rule_predictions = []
changed_ids = []

for row in dev_rows:
    prediction = lemma_majority.get(row["lemma"], "CT+")
    left = row["left"]
    if row["lemma"] == "落ちる" and left.endswith(cue):
        prediction = "NM"
        changed_ids.append(row["id"])
    rule_predictions.append(prediction)

matched = count_agreements(dev_labels, rule_predictions)
print("Cue:", cue)
print("Changed passages:", changed_ids)
print("Validation agreements:", matched, "/", len(dev_rows))

Now read `dev-0046` (「本国の土佐へ【落ち】て行った」) in context. Write down what your rule predicts and what label you choose before revealing Qwen’s label.

In [ ]:
show_passage(by_id["dev-0046"])

In [ ]:
print("Qwen:", by_id["dev-0046"]["reference"])

### Section 5A. Train a character n-gram classifier and tune the window
A character n-gram classifier turns the characters around `【target】` into features (`min_df=2` keeps chunks that appear in at least two training passages). **Edit 2:** Run with `window = 10`, then try `window = 0` (verb only) and `window = 200` (whole sentence). Predict whether narrowing or widening the window helps before rerunning, inspect the top cues, and return `window = 10` (and rerun) before continuing to 5B and 6.

In [ ]:
window = 10

In [ ]:
def make_window(row, width):
    if width <= 0:
        return "【" + row["target"] + "】"
    left_part = row["left"][-width:]
    right_part = row["right"][:width]
    return left_part + "【" + row["target"] + "】" + right_part

train_texts = []
for row in train_rows:
    train_texts.append(make_window(row, window))

dev_texts = []
for row in dev_rows:
    dev_texts.append(make_window(row, window))

vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(1, 3), min_df=2)
x_train_char = vectorizer.fit_transform(train_texts)
x_dev_char = vectorizer.transform(dev_texts)

char_model = LogisticRegression(max_iter=2000, C=1.0)
char_model.fit(x_train_char, train_labels)

char_dev_preds = char_model.predict(x_dev_char)
print("Window:", window, "| Validation agreements:", count_agreements(dev_labels, char_dev_preds), "/", len(dev_rows))
show_scores("Char n-grams (val)", dev_labels, char_dev_preds)
show_confusion_matrix(dev_labels, char_dev_preds)
show_top_cues(vectorizer, char_model, "CT+")
show_top_cues(vectorizer, char_model, "NM")

### Section 5B. Probe minimal pairs and a disputed training passage
First predict the classifier’s labels for the stair minimal pair from the class page (`階段を【上っ】た。` vs. `階段を【上ろ】うとした。`). Then read 「畜犬談」 (`train-0281`), where Qwen chose `NM` and one reader chose `CT+`, and compare movement vs. posture with `垂れる`. **Edit 3:** Change `my_sentence` and test your own prediction.

In [ ]:
show_passage(by_id["train-0281"])

In [ ]:
test_sentence(char_model, vectorizer, "階段を【上っ】た。", window=window)
print()
test_sentence(char_model, vectorizer, "階段を【上ろ】うとした。", window=window)
print()
test_sentence(char_model, vectorizer, "犬は首を【垂れ】た。", window=window)
print()
test_sentence(char_model, vectorizer, "枝から縄が【垂れ】ている。", window=window)

In [ ]:
my_sentence = "犬は首を【垂れ】ている。"
test_sentence(char_model, vectorizer, my_sentence, window=window)

### Section 5C. Train on embeddings, vary training size, and compare weights
Now fit logistic regression on the 1,024-number Qwen embeddings. **Edit 4:** Try `n_train = 32`, `128`, and `420`, compare `C_value = 0.1` vs. `1.0`, and test `use_balanced = True` vs. `False`. Record how macro-F1 and the confusion matrix respond, then return `n_train = 420`, `C_value = 1.0`, and `use_balanced = False` before section 6.

In [ ]:
n_train = 420
C_value = 1.0
use_balanced = False

In [ ]:
sub_vectors = shuffled_vectors[:n_train]
sub_labels = shuffled_labels[:n_train]
weights = "balanced" if use_balanced else None

embed_model = LogisticRegression(max_iter=2000, C=C_value, class_weight=weights)
embed_model.fit(sub_vectors, sub_labels)
embed_dev_preds = embed_model.predict(dev_vectors)

print("Embeddings only (n =", n_train, "C =", C_value, "balanced =", use_balanced, "):", count_agreements(dev_labels, embed_dev_preds), "/", len(dev_rows))
show_scores("Embeddings (val)", dev_labels, embed_dev_preds)
show_confusion_matrix(dev_labels, embed_dev_preds)

x_train_hybrid, cue_encoder = build_hybrid_features(train_rows, train_vectors)
x_dev_hybrid, cue_encoder_dev = build_hybrid_features(dev_rows, dev_vectors, cue_encoder)
hybrid_model = LogisticRegression(max_iter=2000, C=1.0)
hybrid_model.fit(x_train_hybrid, train_labels)
hybrid_dev_preds = hybrid_model.predict(x_dev_hybrid)
show_scores("Embeddings + ending cues (val)", dev_labels, hybrid_dev_preds)

**Edit 5.** Reproduce the website’s NM reading-list slider on the 59 validation passages. Predict how `on_list` and `nm_found` will change when you lower `nm_cutoff` from `0.50` to `0.30`, then change `nm_cutoff` and rerun.

In [ ]:
nm_cutoff = 0.50

In [ ]:
nm_index = list(embed_model.classes_).index("NM")
dev_probs = embed_model.predict_proba(dev_vectors)

on_list = 0
nm_found = 0
nm_total = 0

for row, probs in zip(dev_rows, dev_probs):
    is_nm = row["reference"] == "NM"
    if is_nm:
        nm_total += 1
    if probs[nm_index] >= nm_cutoff:
        on_list += 1
        if is_nm:
            nm_found += 1

print("Cutoff:", nm_cutoff)
print("Passages to read:", on_list, "/", len(dev_rows))
print("NM found:", nm_found, "/", nm_total)

### Section 6. Compare on held-out works
Record your chosen model setting, NM cutoff, and one validation result that supports your choice. Then run the comparison on the 150 held-out passages. Each line reports total agreements, macro-F1, and per-label recall for `NM` and `Uu`.

In [ ]:
test_texts = []
for row in test_rows:
    test_texts.append(make_window(row, window))

char_test_preds = char_model.predict(vectorizer.transform(test_texts))
embed_test_preds = embed_model.predict(test_vectors)
x_test_hybrid, cue_encoder_test = build_hybrid_features(test_rows, test_vectors, cue_encoder)
hybrid_test_preds = hybrid_model.predict(x_test_hybrid)

rules_test_preds = saved_list("rules", test_rows)
gliner_ft_test_preds = saved_list("gliner-ft", test_rows)
lora_2b_test_preds = saved_list("lora-2b", test_rows)

show_scores("Class 4 ending rules", test_labels, rules_test_preds)
show_scores("Char n-grams (your window)", test_labels, char_test_preds)
show_scores("Embeddings (your setting)", test_labels, embed_test_preds)
show_scores("Embeddings + ending cues", test_labels, hybrid_test_preds)
show_scores("GLiNER2 (fine-tuned, GPU)", test_labels, gliner_ft_test_preds)
show_scores("Qwen3.5-2B (LoRA, GPU)", test_labels, lora_2b_test_preds)

### Section 7. Read a test disagreement before revealing labels
Follow the class page by inspecting the `Uu` test disagreements for `lora-2b` (`method_to_check = "lora-2b"`, `reference_filter = "Uu"`), then compare with `method_to_check = "rules"` (or set `reference_filter = "ALL"` to inspect all disagreements). **Edit 6:** Choose `method_to_check`, `reference_filter`, and `case_number`, read the passage, and record your own label and supporting phrase before running the reveal cell.

In [ ]:
method_to_check = "lora-2b"
reference_filter = "Uu"
case_number = 0

In [ ]:
method_preds = {
    "rules": rules_test_preds,
    "char": list(char_test_preds),
    "embed": list(embed_test_preds),
    "gliner-ft": gliner_ft_test_preds,
    "lora-2b": lora_2b_test_preds,
}
chosen_preds = method_preds[method_to_check]

disagreements = []
for idx, row in enumerate(test_rows):
    pred = chosen_preds[idx]
    matches_filter = (reference_filter == "ALL") or (row["reference"] == reference_filter)
    if pred != row["reference"] and matches_filter:
        disagreements.append((row, char_test_preds[idx], embed_test_preds[idx], rules_test_preds[idx], lora_2b_test_preds[idx]))

print("Method:", method_to_check, "| Filter:", reference_filter, "| Disagreements:", len(disagreements))
show_confusion_matrix(test_labels, chosen_preds)

In [ ]:
case_row, case_char_pred, case_embed_pred, case_rules_pred, case_lora_pred = disagreements[case_number]
show_passage(case_row)

Reveal the labels only after recording your own reading and supporting phrase. Which label can you justify from the surrounding sentences?

In [ ]:
print("Qwen (provisional):", case_row["reference"])
print("Class 4 rules:     ", case_rules_pred)
print("Char classifier:   ", case_char_pred)
print("Embed classifier:  ", case_embed_pred)
print("Qwen3.5-2B LoRA:   ", case_lora_pred)